## Merging & Joining DataFrames

### Why does this matter?

In the real world, data never lives in one table. Customer info is in one table, their orders in another, product details in a third. To answer any meaningful business question you have to *combine* these tables. This is exactly what SQL JOINs do — and Pandas gives you the same power with **merge(), concat(), and join()**.

This topic is *heavily tested* in interviews because it directly mirrors SQL knowledge — and every DA job expects you to know both.

### The core idea — what does merging mean?

Merging means combining two DataFrames by matching rows based on a *common column or index*. Think of it like a lookup — "for each row in the left table, find the matching row in the right table and attach its columns."

### Why are there different join types?

Because real data is messy. Sometimes a customer exists in one table but not the other. Different join types give you control over what happens in those mismatch cases — do you keep the unmatched rows or drop them?

In [1]:
import pandas as pd
import numpy as np

# Employee table
employees = pd.DataFrame({
    'EmpID': [1, 2, 3, 4],
    'Name':  ['Sumed', 'Priya', 'Rahul', 'Neha'],
    'DeptID':[10, 20, 10, 30]
})

# Department table
departments = pd.DataFrame({
    'DeptID':   [10, 20, 40],    # Note: 30 missing, 40 extra
    'DeptName': ['Data', 'HR', 'Finance']
})

In [2]:
employees

,EmpID,Name,DeptID
0,1,Sumed,10
1,2,Priya,20
2,3,Rahul,10
3,4,Neha,30


In [3]:
departments

,DeptID,DeptName
0,10,Data
1,20,HR
2,40,Finance


*DeptID 30 (Neha) exists in employees but NOT in departments. DeptID 40 (Finance) exists in departments but NOT in employees. This mismatch is what the 4 join types handle differently.*

## INNER JOIN: only matching rows

In [5]:
# INNER JOIN → keeps ONLY rows where key exists in BOTH tables
# Non-matching rows from either side are dropped
# This is the DEFAULT join type in pd.merge()

pd.merge(employees, departments, on='DeptID', how='inner')

# Neha (DeptID 30) → dropped — 30 not in departments
# Finance (DeptID 40) → dropped — 40 not in employees
# Only DeptID 10 and 20 survive → Sumed, Priya, Rahul

# SQL equivalent:
# SELECT * FROM employees INNER JOIN departments ON employees.DeptID = departments.DeptID

,EmpID,Name,DeptID,DeptName
0,1,Sumed,10,Data
1,2,Priya,20,HR
2,3,Rahul,10,Data


*Inner join is the most common join. Use it when you only want complete, matched records — like joining orders with products where both must exist.*

## LEFT JOIN : all left rows : matching right

In [6]:
# LEFT JOIN → keeps ALL rows from LEFT table
# For left rows with no match in right → fills NaN
# Right rows with no match are dropped

pd.merge(employees, departments, on='DeptID', how='left')

# Neha (DeptID 30) → KEPT, but DeptName = NaN (no match in depts)
# Finance (DeptID 40) → DROPPED (it's on the right side)
# All 4 employees appear in output

# WHEN to use left join?
# "Give me all employees, and their department name if available"
# You never want to lose employees just because dept info is missing

,EmpID,Name,DeptID,DeptName
0,1,Sumed,10,Data
1,2,Priya,20,HR
2,3,Rahul,10,Data
3,4,Neha,30,NaN


*Left join is the second most common join after inner. In real projects, left join is often preferred over inner — you want to know about missing matches (the NaN rows) rather than silently losing data.*

## RIGHT JOIN: allright rows: matching left

In [8]:
# RIGHT JOIN → keeps ALL rows from RIGHT table
# For right rows with no match in left → fills NaN
# Left rows with no match are dropped

pd.merge(employees, departments, on='DeptID', how='right')

# Finance (DeptID 40) → KEPT, but employee columns = NaN
# Neha (DeptID 30) → DROPPED (it's on the left side)
# All 3 departments appear in output

# WHEN to use right join?
# "Give me all departments, even if no employee is assigned"
# Useful for finding empty departments



,EmpID,Name,DeptID,DeptName
0,1.0,Sumed,10,Data
1,3.0,Rahul,10,Data
2,2.0,Priya,20,HR
3,NaN,NaN,40,Finance


In [9]:
# NOTE: right join is rarely used in practice
# You can always get the same result by swapping tables and using left join
pd.merge(departments, employees, on='DeptID', how='left')  # same result

,DeptID,DeptName,EmpID,Name
0,10,Data,1.0,Sumed
1,10,Data,3.0,Rahul
2,20,HR,2.0,Priya
3,40,Finance,NaN,NaN


*EmpID becomes float (1.0, 2.0) because NaN is a float*

## OUTER JOIN : keep all rows from both sides

In [10]:
# OUTER JOIN (full outer) → keeps ALL rows from BOTH tables
# Non-matching rows from either side get NaN for missing columns
# Nothing is dropped — most complete result

pd.merge(employees, departments, on='DeptID', how='outer')

# Neha (DeptID 30) → KEPT from left, DeptName = NaN
# Finance (DeptID 40) → KEPT from right, employee cols = NaN
# All 4 employees AND all 3 departments appear

# WHEN to use outer join?
# When you want to see EVERYTHING — mismatches on both sides
# Great for DATA AUDIT — finding orphaned records on either side

,EmpID,Name,DeptID,DeptName
0,1.0,Sumed,10,Data
1,3.0,Rahul,10,Data
2,2.0,Priya,20,HR
3,4.0,Neha,30,NaN
4,NaN,NaN,40,Finance


## Merging on different column names

In [11]:
# Real world: the key column has different names in each table
# employees has 'DeptID', departments has 'Dept_Code'
# Use left_on and right_on to specify each separately

dept2 = departments.rename(columns={'DeptID': 'Dept_Code'})

pd.merge(
    employees, dept2,
    left_on='DeptID',      # key column in LEFT table
    right_on='Dept_Code',  # key column in RIGHT table
    how='inner'
)
# Both columns appear in output — you can drop one after if needed
# df.drop(columns='Dept_Code')

,EmpID,Name,DeptID,Dept_Code,DeptName
0,1,Sumed,10,10,Data
1,2,Priya,20,20,HR
2,3,Rahul,10,10,Data


*Both key columns appear in the result. Drop the redundant one with drop(columns='Dept_Code') to keep your output clean.*

## Handling duplicate column name with suffixes

In [13]:
# When both tables have a column with the SAME name (not the key)
# Pandas adds suffixes _x and _y to distinguish them

emp = pd.DataFrame({'ID':[1,2], 'Name':['Sumed','Priya'], 'Score':[90,85]})
rev = pd.DataFrame({'ID':[1,2], 'Score':[88,92]})  # Score exists in both

pd.merge(emp, rev, on='ID')
# Score_x → from left table (emp)
# Score_y → from right table (rev)



,ID,Name,Score_x,Score_y
0,1,Sumed,90,88
1,2,Priya,85,92


In [14]:
# Custom suffixes — more meaningful than _x and _y
pd.merge(emp, rev, on='ID', suffixes=('_original', '_revised'))
# Score_original and Score_revised

,ID,Name,Score_original,Score_revised
0,1,Sumed,90,88
1,2,Priya,85,92


## CONCATE : Stacking DataFrames

In [15]:
# concat() is DIFFERENT from merge()
# merge() → combines columns based on matching keys (horizontal logic)
# concat() → stacks DataFrames on top or side by side (no key needed)

# Stack ROWS — axis=0 (default)
# Use when same columns, more rows (e.g. Jan + Feb + Mar data)
jan = pd.DataFrame({'Name':['Sumed','Priya'], 'Sales':[100,200]})
feb = pd.DataFrame({'Name':['Rahul','Neha'],  'Sales':[150,180]})

pd.concat([jan, feb])                       # index keeps original labels
pd.concat([jan, feb], ignore_index=True)   # resets index to 0,1,2,3

# Stack COLUMNS — axis=1
# Use when same rows, more columns (e.g. adding a new feature set)
extra = pd.DataFrame({'Score':[90, 85]})
pd.concat([jan, extra], axis=1)

,Name,Sales,Score
0,Sumed,100,90
1,Priya,200,85


## MERGW vs CONCATE vs JOIN : when to use?

In [ ]:
# merge() → combine on a KEY column. SQL-JOIN style.
#           Use: employees + departments, orders + customers
pd.merge(df1, df2, on='key_col', how='inner/left/right/outer')

# concat() → stack rows or columns. No key needed.
#            Use: Jan+Feb+Mar data, train+test data
pd.concat([df1, df2], axis=0, ignore_index=True)

# join() → merge on INDEX (not column)
#          Use: when both dfs share a meaningful index
#          Less common than merge() in practice
df1.join(df2, how='left')

# REAL WORLD RULE:
# 80% of the time → pd.merge()
# 15% of the time → pd.concat()
# 5% of the time → join()

### Quick decision guide:

Combining two tables that share a common ID column → merge()

Stacking same-structure data from different time periods → concat(axis=0)

Adding new feature columns to existing rows → concat(axis=1)

Joining on index labels → join()

### Everything You Must Know Cold

merge() — combines two DataFrames on a matching key column. 

This is SQL JOIN in Python. The how parameter decides which rows survive.

### Four join types and when to use each:

inner> Only matched rows > You want complete records only

left> All left + matched right> You never want to lose left table rows

right> All right + matched left> You never want to lose right table rows

outer> Everything from both> Data audit — find all mismatches.

## Rule

**left_on / right_on** — when the key column has different names in each table. Both columns appear in output — drop the redundant one after.

**suffixes** — when both tables have a non-key column with the same name, Pandas adds *_x* and *_y*. Override with *suffixes=('_left', '_right')* for clarity.

**concat()** — no key needed. Stack rows with *axis=0* (same columns, more data). Stack columns with *axis=1* (same rows, more features). Always use *ignore_index=True* when stacking rows.

**join()** — merges on index, not columns. Rarely used in practice — *merge()* covers most cases.